In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [36]:
from dataclasses import dataclass

@dataclass
class TPAConfig:
    n_embd:int = 512,
    n_head:int = 8,
    head_dim:int = 64,
    rank:int = 4,
    q_rank:int = 4,
    rope_partial_factor:int = 1

config = TPAConfig()
print(config.n_embd[0])

512


In [49]:
class CPLinear(nn.Module):
    def __init__(self, config):
        super(CPLinear, self).__init__()
        # self.in_features = config.n_embd 
        # self.n_head = config.n_head
        # self.head_dim = config.head_dim
        # self.rank = config.rank
        # self.q_rank = config.q_rank
        # self.rope_partial_factor = config.rope_partial_factor
        self.in_features = 512
        self.n_head = 8
        self.head_dim = 64
        self.rank = 4
        self.q_rank = 4
        self.rope_partial_factor = 1

        print(self.n_head)
        print(self.head_dim)
        print(self.n_head * self.head_dim)
        
        self.c_q = nn.Linear(self.in_features, self.n_head * self.head_dim, bias=False)
        
        # Define linear transformations for A projections
        self.W_A_k = nn.Linear(self.in_features, self.n_head * self.rank, bias=False)
        self.W_A_v = nn.Linear(self.in_features, self.n_head * self.rank, bias=False)
        
        # Define B projection parameters for K, V
        self.W_B_k = nn.Linear(self.in_features, self.rank * self.head_dim, bias=False)
        self.W_B_v = nn.Linear(self.in_features, self.rank * self.head_dim, bias=False)
        # Calculate the dimension for RoPE based on the partial factor
        rotary_dim = int(self.head_dim * self.rope_partial_factor)
        # Ensure the rotary dimension is even
        rotary_dim = (rotary_dim // 2) * 2
        # self.rotary = RotaryDecay(rotary_dim, decay_base=config.rope_decay_base)
        self.reset_parameters()

    def reset_parameters(self):
        W_A_k_tensor = self.W_A_k.weight.view(self.in_features, self.n_head, self.rank)
        W_A_v_tensor = self.W_A_v.weight.view(self.in_features, self.n_head, self.rank)
        nn.init.xavier_uniform_(W_A_k_tensor)
        nn.init.xavier_uniform_(W_A_v_tensor)
        self.W_A_k.weight.data = W_A_k_tensor.view_as(self.W_A_k.weight)
        self.W_A_v.weight.data = W_A_v_tensor.view_as(self.W_A_v.weight)
        
        W_B_k_tensor = self.W_B_k.weight.view(self.in_features, self.rank, self.head_dim)
        W_B_v_tensor = self.W_B_v.weight.view(self.in_features, self.rank, self.head_dim)
        nn.init.xavier_uniform_(W_B_k_tensor)
        nn.init.xavier_uniform_(W_B_v_tensor)
        self.W_B_k.weight.data = W_B_k_tensor.view_as(self.W_B_k.weight)
        self.W_B_v.weight.data = W_B_v_tensor.view_as(self.W_B_v.weight)
		
        nn.init.xavier_uniform_(self.c_q.weight)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        
        q = self.c_q(x).view(batch_size, seq_len, self.n_head, self.head_dim)
        
        # Compute intermediate variables A for K, and V
        A_k = self.W_A_k(x).view(batch_size, seq_len, self.n_head, self.rank)
        A_v = self.W_A_v(x).view(batch_size, seq_len, self.n_head, self.rank)
        
        # Compute intermediate variables B for K, and V
        B_k = self.W_B_k(x).view(batch_size, seq_len, self.rank, self.head_dim)
        B_v = self.W_B_v(x).view(batch_size, seq_len, self.rank, self.head_dim)

        # if is_apply_rope:
        #     # Apply rotary embeddings
        #     cos, sin, decay = self.rotary(q)
        #     # q, k = F.rms_norm(q, (q.size(-1),)), F.rms_norm(k, (k.size(-1),)) # QK rms norm
        #     q, B_k = apply_rotary_emb(q, cos, sin, decay), apply_rotary_emb(B_k, cos, sin, decay)

        
        # Reshape A_k, A_v
        A_k = A_k.view(batch_size * seq_len, self.n_head, self.rank)
        A_v = A_v.view(batch_size * seq_len, self.n_head, self.rank)
        
        # Reshape B_k, B_v  
        B_k = B_k.view(batch_size * seq_len, self.rank, self.head_dim)
        B_v = B_v.view(batch_size * seq_len, self.rank, self.head_dim)
        
        k = torch.bmm(A_k, B_k).div_(self.rank).view(batch_size, seq_len, self.n_head, self.head_dim)
        v = torch.bmm(A_v, B_v).div_(self.rank).view(batch_size, seq_len, self.n_head, self.head_dim)
        
        return q, k, v

In [50]:
linear = CPLinear(config, )
linear.reset_parameters()

8
64
512


In [51]:
bs = 6
seq_len = 7
dim = 512

X = torch.randn(bs, seq_len, dim)

In [52]:
q, k, v = linear(X)
print(q.shape)
print(k.shape)
print(v.shape)

torch.Size([6, 7, 8, 64])
torch.Size([6, 7, 8, 64])
torch.Size([6, 7, 8, 64])
